# 02 — PatchCore Memory Bank Creation

This notebook builds PatchCore anomaly detection models for each product category.

**How PatchCore works:**
1. Extract patch-level features from good images using a pretrained WideResNet-50
2. Store a coreset-subsampled memory bank of normal features
3. At test time, anomaly score = nearest-neighbor distance to memory bank
4. High distance = anomaly — no training required, just feature extraction

**Contents:**
1. Setup and data loading
2. Feature extraction and memory bank creation
3. Threshold computation from validation set
4. Visualize anomaly score distributions
5. Quick test: good vs defective comparison

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from src.dataset import MVTecDataset, create_train_val_split
from src.model import PatchCoreModel, get_device

# Configuration
DATA_ROOT = "../data/raw"
CATEGORIES = ["bottle", "cable", "transistor"]
CORESET_RATIO = 0.25
SIGMA = 1.0

device = get_device()
print(f"Using device: {device}")

## 1. Data Loading

In [ ]:
# Show dataset stats for each category
for category in CATEGORIES:
    train_ds, val_ds = create_train_val_split(DATA_ROOT, category, val_ratio=0.2, seed=42)
    test_ds = MVTecDataset(DATA_ROOT, category, split="test")
    counts = test_ds.get_class_counts()
    print(f"\n{category.upper()}:")
    print(f"  Train: {len(train_ds)} good images | Val: {len(val_ds)} good images")
    print(f"  Test:  {counts['good']} good + {counts['defective']} defective")

# Show sample training images from bottle
train_ds, _ = create_train_val_split(DATA_ROOT, "bottle", val_ratio=0.2, seed=42)
from torch.utils.data import DataLoader
loader = DataLoader(train_ds, batch_size=8, shuffle=True)
images, _ = next(iter(loader))

fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for i in range(8):
    img = images[i].permute(1, 2, 0).numpy()
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])  # denormalize
    axes[i].imshow(np.clip(img, 0, 1))
    axes[i].axis("off")
fig.suptitle("Sample Training Images — Bottle (all good)", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Build PatchCore Models

For each category: extract features from good training images, build a memory bank, and compute a threshold from the validation set. No gradient-based training needed!

In [ ]:
from src.visualize import compute_anomaly_heatmap
from torch.utils.data import DataLoader
from PIL import Image

models = {}

for category in CATEGORIES:
    print(f"\n{'='*50}")
    print(f"Building PatchCore model for: {category.upper()}")
    print(f"{'='*50}")
    
    # Create train/val split
    train_ds, val_ds = create_train_val_split(DATA_ROOT, category, val_ratio=0.2, seed=42)
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=False, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)
    
    # Build PatchCore model
    model = PatchCoreModel(device=device)
    
    # Step 1: Extract features and build memory bank
    print(f"Extracting features from {len(train_ds)} good images...")
    model.fit(train_loader, coreset_ratio=CORESET_RATIO)
    print(f"Memory bank: {model.memory_bank.shape[0]} patches")
    
    # Step 2: Compute threshold from validation set
    print("Computing threshold from validation set...")
    val_scores = []
    for images, _ in val_loader:
        for img_tensor in images:
            score, _ = model.predict(img_tensor.unsqueeze(0))
            val_scores.append(score)
    
    val_scores = np.array(val_scores)
    model.threshold = float(val_scores.mean() + SIGMA * val_scores.std())
    print(f"Threshold: {model.threshold:.4f} (mean={val_scores.mean():.4f}, std={val_scores.std():.4f})")
    
    # Save model
    save_path = f"../models/best_{category}.pth"
    model.save(save_path)
    models[category] = model
    print(f"Saved to {save_path}")

print("\nAll models built!")

## 3. Validation Score Distributions

The threshold is set to separate good images (low scores) from potential anomalies (high scores).

In [ ]:
# Plot validation score distributions for all categories
fig, axes = plt.subplots(1, len(CATEGORIES), figsize=(6 * len(CATEGORIES), 5))

for i, category in enumerate(CATEGORIES):
    model = models[category]
    val_ds = create_train_val_split(DATA_ROOT, category, val_ratio=0.2, seed=42)[1]
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)
    
    scores = []
    for images, _ in val_loader:
        for img_tensor in images:
            score, _ = model.predict(img_tensor.unsqueeze(0))
            scores.append(score)
    
    axes[i].hist(scores, bins=20, alpha=0.7, color="green", edgecolor="black")
    axes[i].axvline(model.threshold, color="red", linestyle="--", label=f"Threshold: {model.threshold:.4f}")
    axes[i].set_title(f"{category.capitalize()} — Validation Scores (good only)")
    axes[i].set_xlabel("Anomaly Score")
    axes[i].set_ylabel("Count")
    axes[i].legend()

plt.tight_layout()
plt.show()

## 4. Quick Test: Good vs Defective

Compare anomaly heatmaps on good and defective test images to verify the model works.

In [ ]:
for category in CATEGORIES:
    model = models[category]
    test_ds = MVTecDataset(DATA_ROOT, category, split="test")
    
    # Find one good and one defective image
    good_idx = next(i for i, l in enumerate(test_ds.labels) if l == 0)
    bad_idx = next(i for i, l in enumerate(test_ds.labels) if l == 1)
    
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    fig.suptitle(f"{category.capitalize()} — Good vs Defective", fontsize=14)
    
    for row, idx, label in [(0, good_idx, "GOOD"), (1, bad_idx, "DEFECTIVE")]:
        image = Image.open(test_ds.image_paths[idx]).convert("RGB")
        heatmap, score = compute_anomaly_heatmap(model, image)
        overlay = __import__("src.visualize", fromlist=["create_overlay"]).create_overlay(image, heatmap)
        
        verdict = "PASS" if score <= model.threshold else "DEFECT"
        
        axes[row, 0].imshow(image.resize((224, 224)))
        axes[row, 0].set_title(f"{label} — Original")
        axes[row, 0].axis("off")
        
        axes[row, 1].imshow(overlay)
        axes[row, 1].set_title(f"Heatmap — Score: {score:.4f} ({verdict})")
        axes[row, 1].axis("off")
    
    plt.tight_layout()
    plt.show()